# Task 1: Text Classification (SST-2)
**CENG 467 — NLU&G Take-Home Midterm | Student ID: 300201071**

Models: TF-IDF + LinearSVC, BiLSTM, DistilBERT

Metrics: Accuracy, Macro-F1

In [1]:
!pip install -q datasets transformers scikit-learn nltk torch

In [2]:
import os, random, re, string, json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from tqdm.auto import tqdm
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
from datasets import load_dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset as HFDataset

# ========== REPRODUCIBILITY ==========
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: Tesla T4


## 1. Load SST-2 Dataset

In [3]:
dataset = load_dataset('glue', 'sst2')
test_data = dataset['validation']  # Official val -> our test (labels hidden in GLUE test)
train_val = dataset['train'].train_test_split(test_size=0.1, seed=SEED)
train_data = train_val['train']
dev_data = train_val['test']
print(f'Train: {len(train_data)}, Dev: {len(dev_data)}, Test: {len(test_data)}')

train_texts = train_data['sentence']
train_labels = train_data['label']
dev_texts = dev_data['sentence']
dev_labels = dev_data['label']
test_texts = test_data['sentence']
test_labels = test_data['label']

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Train: 60614, Dev: 6735, Test: 872


## 2. Preprocessing Pipeline — Normalization Comparison

In [4]:
STOP_WORDS = set(stopwords.words('english'))

def normalize_raw(text): return text

def normalize_lower_nopunct(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return re.sub(r'\s+', ' ', text).strip()

def normalize_lower_nostop(text):
    text = text.lower().translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    return ' '.join([t for t in tokens if t not in STOP_WORDS])

## 3. Model A: TF-IDF + LinearSVC

In [5]:
normalizers = {'raw': normalize_raw, 'lower_nopunct': normalize_lower_nopunct, 'lower_nostop': normalize_lower_nostop}
norm_results = {}
best_acc, best_norm, best_test_preds = 0, '', None

for name, fn in normalizers.items():
    tr = [fn(t) for t in train_texts]
    dv = [fn(t) for t in dev_texts]
    ts = [fn(t) for t in test_texts]
    pipe = Pipeline([('tfidf', TfidfVectorizer(max_features=50000, ngram_range=(1,2), sublinear_tf=True)),
                     ('svc', LinearSVC(C=1.0, max_iter=10000, random_state=SEED))])
    pipe.fit(tr, train_labels)
    dev_preds = pipe.predict(dv)
    acc = accuracy_score(dev_labels, dev_preds)
    f1 = f1_score(dev_labels, dev_preds, average='macro')
    norm_results[name] = {'dev_acc': acc, 'dev_f1': f1}
    print(f'  {name:20s}  Dev Acc={acc:.4f}  Dev F1={f1:.4f}')
    if acc > best_acc:
        best_acc, best_norm = acc, name
        pipe2 = Pipeline([('tfidf', TfidfVectorizer(max_features=50000, ngram_range=(1,2), sublinear_tf=True)),
                          ('svc', LinearSVC(C=1.0, max_iter=10000, random_state=SEED))])
        pipe2.fit(tr, train_labels)
        best_test_preds = pipe2.predict(ts)

tfidf_acc = accuracy_score(test_labels, best_test_preds)
tfidf_f1 = f1_score(test_labels, best_test_preds, average='macro')
print(f'\nBest normalization: {best_norm}')
print(f'TF-IDF+SVC TEST => Accuracy: {tfidf_acc:.4f}, Macro-F1: {tfidf_f1:.4f}')

  raw                   Dev Acc=0.9206  Dev F1=0.9196
  lower_nopunct         Dev Acc=0.9201  Dev F1=0.9192
  lower_nostop          Dev Acc=0.9201  Dev F1=0.9192

Best normalization: raw
TF-IDF+SVC TEST => Accuracy: 0.8131, Macro-F1: 0.8125


## 4. Model B: BiLSTM

In [6]:
def tokenize_wl(text): return word_tokenize(text.lower())

counter = Counter()
for t in train_texts: counter.update(tokenize_wl(t))
vocab = {'<pad>': 0, '<unk>': 1}
for w, c in counter.most_common():
    if c >= 2: vocab[w] = len(vocab)
print(f'Vocab size: {len(vocab)}')

MAX_LEN = 128
def to_ids(texts, v, ml):
    r = np.zeros((len(texts), ml), dtype=np.int64)
    for i, t in enumerate(texts):
        for j, w in enumerate(tokenize_wl(t)[:ml]):
            r[i, j] = v.get(w, v['<unk>'])
    return r

X_tr = to_ids(train_texts, vocab, MAX_LEN)
X_dv = to_ids(dev_texts, vocab, MAX_LEN)
X_ts = to_ids(test_texts, vocab, MAX_LEN)
y_tr, y_dv, y_ts = np.array(train_labels), np.array(dev_labels), np.array(test_labels)

class TDS(Dataset):
    def __init__(s, X, y): s.X, s.y = torch.LongTensor(X), torch.LongTensor(y)
    def __len__(s): return len(s.y)
    def __getitem__(s, i): return s.X[i], s.y[i]

BS = 32
tr_dl = DataLoader(TDS(X_tr, y_tr), batch_size=BS, shuffle=True)
dv_dl = DataLoader(TDS(X_dv, y_dv), batch_size=BS)
ts_dl = DataLoader(TDS(X_ts, y_ts), batch_size=BS)

Vocab size: 14123


In [8]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vs, ed=100, hd=256, nc=2, do=0.5):
        super().__init__()
        self.emb = nn.Embedding(vs, ed, padding_idx=0)
        self.lstm = nn.LSTM(ed, hd, batch_first=True, bidirectional=True, num_layers=2, dropout=do)
        self.drop = nn.Dropout(do)
        self.fc = nn.Linear(hd*2, nc)
    def forward(self, x):
        e = self.drop(self.emb(x))
        _, (h, _) = self.lstm(e)
        h = torch.cat((h[-2], h[-1]), dim=1)
        return self.fc(self.drop(h))

model = BiLSTMClassifier(len(vocab)).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()
print(f'BiLSTM params: {sum(p.numel() for p in model.parameters()):,}')

for ep in range(10):
    model.train(); tl = 0
    for xb, yb in tr_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad(); loss = crit(model(xb), yb); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); tl += loss.item()
    model.eval(); vl = 0
    with torch.no_grad():
        for xb, yb in dv_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            vl += crit(model(xb), yb).item()
    print(f'  Ep {ep+1}/10: Train={tl/len(tr_dl):.4f} Val={vl/len(dv_dl):.4f}')

model.eval(); all_p = []
with torch.no_grad():
    for xb, _ in ts_dl:
        all_p.extend(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
bilstm_acc = accuracy_score(y_ts, all_p)
bilstm_f1 = f1_score(y_ts, all_p, average='macro')
print(f'\nBiLSTM TEST => Accuracy: {bilstm_acc:.4f}, Macro-F1: {bilstm_f1:.4f}')

BiLSTM params: 3,723,470
  Ep 1/10: Train=0.6100 Val=0.4726
  Ep 2/10: Train=0.4410 Val=0.3274
  Ep 3/10: Train=0.3496 Val=0.2852
  Ep 4/10: Train=0.2994 Val=0.2608
  Ep 5/10: Train=0.2625 Val=0.2420
  Ep 6/10: Train=0.2343 Val=0.2325
  Ep 7/10: Train=0.2204 Val=0.2219
  Ep 8/10: Train=0.2021 Val=0.2233
  Ep 9/10: Train=0.1839 Val=0.2327
  Ep 10/10: Train=0.1715 Val=0.2242

BiLSTM TEST => Accuracy: 0.8372, Macro-F1: 0.8371


## 5. Model C: DistilBERT

In [9]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
bert_model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

def make_ds(texts, labels):
    enc = tokenizer(list(texts), truncation=True, padding='max_length', max_length=128)
    ds = HFDataset.from_dict({'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask'], 'labels': list(labels)})
    ds.set_format('torch')
    return ds

def compute_m(ep):
    logits, labels = ep
    p = np.argmax(logits, axis=1)
    return {'accuracy': accuracy_score(labels, p), 'macro_f1': f1_score(labels, p, average='macro')}

tr_ds = make_ds(train_texts, train_labels)
dv_ds = make_ds(dev_texts, dev_labels)
ts_ds = make_ds(test_texts, test_labels)

args = TrainingArguments(
    output_dir='/content/distilbert_sst2', num_train_epochs=3,
    per_device_train_batch_size=16, per_device_eval_batch_size=16,
    learning_rate=2e-5, weight_decay=0.01, eval_strategy='epoch',
    save_strategy='epoch', load_best_model_at_end=True,
    metric_for_best_model='macro_f1', seed=SEED, logging_steps=200,
    report_to='none', fp16=torch.cuda.is_available())

trainer = Trainer(model=bert_model, args=args, train_dataset=tr_ds,
                  eval_dataset=dv_ds, compute_metrics=compute_m)
trainer.train()

res = trainer.predict(ts_ds)
bert_preds = np.argmax(res.predictions, axis=1)
bert_acc = accuracy_score(test_labels, bert_preds)
bert_f1 = f1_score(test_labels, bert_preds, average='macro')
print(f'\nDistilBERT TEST => Accuracy: {bert_acc:.4f}, Macro-F1: {bert_f1:.4f}')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



DistilBERT TEST => Accuracy: 0.9060, Macro-F1: 0.9059


## 6. Error Analysis (5 Misclassified Examples per Model)

In [10]:
label_map = {0: 'Negative', 1: 'Positive'}
models_preds = {'TF-IDF+SVC': best_test_preds, 'BiLSTM': all_p, 'DistilBERT': bert_preds}

for mname, preds in models_preds.items():
    print(f'\n{"="*60}')
    print(f'{mname} — Misclassified Examples')
    print(f'{"="*60}')
    count = 0
    for i in range(len(test_labels)):
        if int(test_labels[i]) != int(preds[i]):
            print(f'  [{count+1}] "{test_texts[i][:120]}"')
            print(f'       True: {label_map[int(test_labels[i])]}, Pred: {label_map[int(preds[i])]}')
            count += 1
            if count >= 5: break


TF-IDF+SVC — Misclassified Examples
  [1] "or doing last year 's taxes with your ex-wife . "
       True: Negative, Pred: Positive
  [2] "in its best moments , resembles a bad high school production of grease , without benefit of song . "
       True: Negative, Pred: Positive
  [3] "the iditarod lasts for days - this just felt like it did . "
       True: Negative, Pred: Positive
  [4] "if the movie succeeds in instilling a wary sense of ` there but for the grace of god , ' it is far too self-conscious to"
       True: Negative, Pred: Positive
  [5] "( w ) hile long on amiable monkeys and worthy environmentalism , jane goodall 's wild chimpanzees is short on the thrill"
       True: Negative, Pred: Positive

BiLSTM — Misclassified Examples
  [1] "you do n't have to know about music to appreciate the film 's easygoing blend of comedy and romance . "
       True: Positive, Pred: Negative
  [2] "we root for ( clara and paul ) , even like them , though perhaps it 's an emotion closer to p

## 7. Final Results Summary (for LaTeX Report)

In [11]:
print('\n' + '='*60)
print('  TASK 1 — FINAL RESULTS (Copy to LaTeX)')
print('='*60)
print(f'{"Model":<20} {"Accuracy":<12} {"Macro-F1":<12}')
print('-'*44)
print(f'{"TF-IDF + LinearSVC":<20} {tfidf_acc:<12.4f} {tfidf_f1:<12.4f}')
print(f'{"BiLSTM":<20} {bilstm_acc:<12.4f} {bilstm_f1:<12.4f}')
print(f'{"DistilBERT":<20} {bert_acc:<12.4f} {bert_f1:<12.4f}')
print('-'*44)
print('\nNormalization Comparison (TF-IDF Dev Set):')
for n, r in norm_results.items():
    print(f'  {n:<20} Acc={r["dev_acc"]:.4f}  F1={r["dev_f1"]:.4f}')


  TASK 1 — FINAL RESULTS (Copy to LaTeX)
Model                Accuracy     Macro-F1    
--------------------------------------------
TF-IDF + LinearSVC   0.8131       0.8125      
BiLSTM               0.8372       0.8371      
DistilBERT           0.9060       0.9059      
--------------------------------------------

Normalization Comparison (TF-IDF Dev Set):
  raw                  Acc=0.9206  F1=0.9196
  lower_nopunct        Acc=0.9201  F1=0.9192
  lower_nostop         Acc=0.9201  F1=0.9192
